In [ ]:
import tidy3d as td
import numpy as np
import tidy3d.web as web

import math
import os
from pathlib import Path

# import need be changed in some cases

# --- 1. Material Definitions ---
# Using fixed indices for 1550nm for simplicity

wavelength =  1.55                              # Central wavelength
wv_points = 3                                   # number of wavelength wv_points
wdth_points = 80                                # number of width wv_points
num_modes = 2                                   # max number of modes to look up
n_si = 3.475
n_sio2 = 1.458

sweep_wavelength = np.linspace(1.5,1.6,wv_points)  # Sweep on wavelengths
sweep_freq = td.C_0 / sweep_wavelength          # Sweep on Frequencies
sweep_width = np.linspace(0.220,1,wdth_points)

mat_SiN = td.material_library["SiN"]["Horiba"] # Material trace permitivity model for crystaline Silicon
mat_sio2 = td.material_library["SiO2"]["Horiba"]   # Material trace permitivity model for crystaline Silica


#
# Silicon_Dioxide___SiO2___Horiba = td.PoleResidue(
#     name = 'Silicon Dioxide ("SiO2")_Horiba',
#     frequency_range = [169259246959034.88, 1208994621135963.5],
#     poles = [[(-75963372399806.36-18231051118240810j), (0 + 10209565875622414j)]]
# )
#
# Silicon_Mononitride___SiN___Horiba = td.PoleResidue(
#     name = 'Silicon Mononitride ("SiN")_Horiba',
#     frequency_range = [145079354536315.6, 1450793545363156],
#     eps_inf = 2.32,
#     poles = [[(-302334222151229.3-9863009385232968j), (0 + 6244215164693547j)]]
# )


waveguide = td.Structure(
    geometry = td.Box(size = [td.inf, core_width, core_thickness]),
    name = 'waveguide',
    medium = mat_SiN
)


sim = td.ModeSimulation(
    freqs = [428274940000000, 409788251942446.06, 392831496689655.2, 377222298145695.4, 362806159363057.4, 349451331411043, 337044775266272.25, 325488954400000.06, 314699265303867.44, 304601962673796.8, 295132471606217.7, 286234005125628.2, 277856424487804.88, 269955293933649.25, 262491092258064.47, 255428551659192.75, 248736100524017.38, 242385391574467.97, 236350900497925.2, 230609583076923.06],
    mode_spec = td.ModeSpec(target_neff = 3.475, sort_spec = {'filter_reference' : 0, 'filter_order':'over', 'sort_order':'ascending', 'track_freq':'central'}, group_index_step = True, ),
    size = [7, 7, 7],
    grid_spec = td.GridSpec(grid_x = td.AutoGrid(min_steps_per_wvl = 11, ), grid_y = td.AutoGrid(min_steps_per_wvl = 11, ), grid_z = td.AutoGrid(min_steps_per_wvl = 11, ), wavelength = 0.7, ),
    version = '2.10.1',
    medium = Silicon_Dioxide___SiO2___Horiba,
    sources = [],
    monitors = [],
    structures = [waveguide],
)
sim_data = web.run(sim, task_name='Mode Analysis_MODE', path='./data/sim_data.hdf5')

version_name = "SiN_sim"

project_dir = Path.cwd()  # directory where notebook is located
data_dir = project_dir / "data_hom2_point3"
data_dir.mkdir(parents=True, exist_ok=True)


def build_mode_simulation(
    cose_width = np.array([0.600]),
    core_thickness = 0.220,
    wavelength = np.array([0.7]),
    version_name = "sim_mmi"
):

    base_path = f"data_hom2_point3/{version_name}"
    os.makedirs(base_path, exist_ok=True)

    # Frequency
    freq = td.C_0 / wavelength

    # Materials
    core = mat_si
    clad = mat_sio2

    # --- We define the simulation data array and simulation objects for the two different sweeps----

    sim_data_arr = [[[]],[[]]] # Simulation data for 220nm and 300nm, Length sweep and Width sweep
    sim_arr = [[[]],[[]]]      # Simulation objects for 220nm and 300nm , Length sweep and Width sweep
    estimate = 0

    #--- Define Sources ---
    freq0 = td.C_0 / wavelength.mean()
    fwidth = (td.C_0 / wavelength.min()) - (td.C_0 / wavelength.max())



    for (thick_idx,thick) in enumerate(thickness):
        thick_folder = f"{base_path}/t{int(thick*1000)}"
        os.makedirs(thick_folder, exist_ok=True)

        for (length_idx,length_values) in enumerate(MMI_length):
            length_folder = f"{thick_folder}/L{int(length_values*1000)}"
            os.makedirs(length_folder, exist_ok=True)
            for (width_idx,width_values) in enumerate(MMI_width):

                filename = f"{length_folder}/width_{int(width_values*1000)}.hdf5"

                ## Definicion de la fuente ##
                source1 = td.ModeSource(
                    name = 'Mode_source',
                    center = [Initial_pos-3.5, 0, 0],
                    size = [0, 4, 4],
                    source_time = td.GaussianPulse(freq0 = freq0, fwidth = fwidth, ),
                    direction = '+',
                    mode_spec = td.ModeSpec(num_modes = 1, target_neff = n_si, sort_spec = {'filter_reference' : 0, 'filter_order':'over', 'sort_order':'ascending', 'track_freq':'central'}, group_index_step = True, ),
                )

                ## Definicion de los monitores ##

                Longitudinal = td.FieldMonitor(
                    name = 'Longitudinal',
                    size = [td.inf, 7.75, 0],
                    freqs = freq,
                )

                IN = td.FieldMonitor(
                    name = 'IN',
                    center = [Initial_pos-2.5, 0, 0],
                    size = [0, 4, 4],
                    freqs = freq,
                )

                OUT1 = td.FieldMonitor(
                    name = 'OUT1',
                    center = [length_values + Initial_pos + 3, dinstance_tapers[width_idx] / 2, 0],
                    size = [0, 1, 1],
                    freqs = freq,
                )

                OUT2 = td.FieldMonitor(
                    name = 'OUT2',
                    center = [length_values + Initial_pos + 3, -dinstance_tapers[width_idx] / 2, 0],
                    size = [0, 1, 1],
                    freqs = freq,
                )


                ## Definicion de las estructuras ##


                ## For a Strip
                Xmin_Strip= (-100)
                Xmax_Strip= (-2+Initial_pos)

                Ymin_Strip= (-width_single_mode[thick_idx]/2)
                Ymax_Strip= (width_single_mode[thick_idx]/2)

                Zmin_Strip= (-thick/2)
                Zmax_Strip= (thick/2)

                Strip = td.Structure(
                    geometry = td.Box(center = [(Xmax_Strip+Xmin_Strip)/2, (Ymax_Strip+Ymin_Strip)/2, (Zmax_Strip+Zmin_Strip)/2], size = [Xmax_Strip-Xmin_Strip, Ymax_Strip-Ymin_Strip, Zmax_Strip-Zmin_Strip], ),
                    name = 'Strip',
                    medium = core
                )



                 ## For the MMI

                Xmin_MMI= Initial_pos
                Xmax_MMI= length_values+Initial_pos

                Ymin_MMI= -width_values/2
                Ymax_MMI= width_values/2

                Zmin_MMI= (-thick/2)
                Zmax_MMI= (thick/2)

                MMI = td.Structure(
                    geometry = td.Box(center = [(Xmax_MMI+Xmin_MMI)/2, (Ymax_MMI+Ymin_MMI)/2, (Zmax_MMI+Zmin_MMI)/2], size = [Xmax_MMI-Xmin_MMI, Ymax_MMI-Ymin_MMI, Zmax_MMI-Zmin_MMI]),
                    name = 'MMI',
                    medium = core
                )


                ## For the Taper In

                Xv1_Taper_IN =-2+Initial_pos
                Yv1_Taper_IN =-width_single_mode[thick_idx]/2

                Xv2_Taper_IN = Initial_pos
                Yv2_Taper_IN =-taper_width/2

                Xv3_Taper_IN = Initial_pos
                Yv3_Taper_IN = taper_width/2

                Xv4_Taper_IN = -2 + Initial_pos
                Yv4_Taper_IN = width_single_mode[thick_idx] / 2

                Zmin_Taper_IN= (-thick/2)
                Zmax_Taper_IN= (thick/2)


                Taper_IN = td.Structure(
                    geometry = td.PolySlab(slab_bounds = [Zmin_Taper_IN, Zmax_Taper_IN], vertices = [[Xv1_Taper_IN, Yv1_Taper_IN], [Xv2_Taper_IN, Yv2_Taper_IN], [Xv3_Taper_IN, Yv3_Taper_IN], [Xv4_Taper_IN, Yv4_Taper_IN]]),
                    name = 'Taper',
                    medium = core
                )

                 ## For the Taper OUT 1

                Xv1_Taper_OUT_1 = length_values+Initial_pos+2
                Yv1_Taper_OUT_1 =-width_single_mode[thick_idx]/2-dinstance_tapers[width_idx]/2

                Xv2_Taper_OUT_1 = length_values+Initial_pos
                Yv2_Taper_OUT_1 =-dinstance_tapers[width_idx]/2-taper_width/2

                Xv3_Taper_OUT_1 = length_values+Initial_pos
                Yv3_Taper_OUT_1 = -dinstance_tapers[width_idx]/2+taper_width/2

                Xv4_Taper_OUT_1 = length_values+Initial_pos+2
                Yv4_Taper_OUT_1 = -dinstance_tapers[width_idx]/2+width_single_mode[thick_idx]/2

                Zmin_Taper_OUT_1= (-thick/2)
                Zmax_Taper_OUT_1= (thick/2)



                Taper_out1 = td.Structure(
                    geometry = td.PolySlab(slab_bounds = [Zmin_Taper_OUT_1, Zmax_Taper_OUT_1], vertices = [[Xv1_Taper_OUT_1, Yv1_Taper_OUT_1], [Xv2_Taper_OUT_1, Yv2_Taper_OUT_1], [Xv3_Taper_OUT_1, Yv3_Taper_OUT_1], [Xv4_Taper_OUT_1, Yv4_Taper_OUT_1]]),
                    name = 'Taper_out1',
                    medium = core
                )


                ## For the Taper OUT 2

                Xv1_Taper_OUT_2 = length_values+Initial_pos+2
                Yv1_Taper_OUT_2 =width_single_mode[thick_idx]/2+dinstance_tapers[width_idx]/2

                Xv2_Taper_OUT_2 = length_values+Initial_pos
                Yv2_Taper_OUT_2 =dinstance_tapers[width_idx]/2+taper_width/2

                Xv3_Taper_OUT_2 = length_values+Initial_pos
                Yv3_Taper_OUT_2 = dinstance_tapers[width_idx]/2-taper_width/2

                Xv4_Taper_OUT_2 = length_values+Initial_pos+2
                Yv4_Taper_OUT_2 = dinstance_tapers[width_idx]/2-width_single_mode[thick_idx]/2

                Zmin_Taper_OUT_2= (-thick/2)
                Zmax_Taper_OUT_2= (thick/2)


                Taper_out2 = td.Structure(
                    geometry = td.PolySlab(slab_bounds = [Zmin_Taper_OUT_2, Zmax_Taper_OUT_2], vertices = [[Xv1_Taper_OUT_2, Yv1_Taper_OUT_2], [Xv2_Taper_OUT_2, Yv2_Taper_OUT_2], [Xv3_Taper_OUT_2, Yv3_Taper_OUT_2], [Xv4_Taper_OUT_2, Yv4_Taper_OUT_2]]),
                    name = 'Taper_out2',
                    medium = core
                )


                ## for a Strip_out1

                Xmin_Strip_out1= length_values+Initial_pos+2
                Xmax_Strip_out1= length_values+100

                Ymin_Strip_out1= -dinstance_tapers[width_idx]/2-width_single_mode[thick_idx]/2
                Ymax_Strip_out1= width_single_mode[thick_idx]/2-dinstance_tapers[width_idx]/2

                Zmin_Strip_out1= (-thick/2)
                Zmax_Strip_out1= (thick/2)



                Strip_out1 = td.Structure(
                    geometry = td.Box(center = [(Xmax_Strip_out1+Xmin_Strip_out1)/2, (Ymax_Strip_out1+Ymin_Strip_out1)/2, (Zmax_Strip_out1+Zmin_Strip_out1)/2], size = [Xmax_Strip_out1-Xmin_Strip_out1, Ymax_Strip_out1-Ymin_Strip_out1, Zmax_Strip_out1-Zmin_Strip_out1]),
                    name = 'Strip_out1',
                    medium = core
                )

                ## for a Strip_out2

                Xmin_Strip_out2= length_values+Initial_pos+2
                Xmax_Strip_out2= length_values+100

                Ymin_Strip_out2= dinstance_tapers[width_idx]/2-width_single_mode[thick_idx]/2
                Ymax_Strip_out2= width_single_mode[thick_idx]/2+dinstance_tapers[width_idx]/2

                Zmin_Strip_out2= (-thick/2)
                Zmax_Strip_out2= (thick/2)


                Strip_out2 = td.Structure(
                    geometry = td.Box(center = [(Xmax_Strip_out2+Xmin_Strip_out2)/2, (Ymax_Strip_out2+Ymin_Strip_out2)/2, (Zmax_Strip_out2+Zmin_Strip_out2)/2], size = [Xmax_Strip_out2-Xmin_Strip_out2, Ymax_Strip_out2-Ymin_Strip_out2, Zmax_Strip_out2-Zmin_Strip_out2]),
                    name = 'Strip_out2',
                    medium = core
                )#Hasta aca todo bien



                # --- Simulation domain ---
                sim_arr[thick_idx][length_idx].append(td.Simulation(
                    center = [-3.5+(x_window-23)/2, 0, 0],
                    size=[x_window, 6, 6],  # x ignored for mode solver
                    monitors=[OUT1,OUT2,Longitudinal,IN],
                    sources=[source1],
                    structures=[Strip,Strip_out1,Strip_out2,MMI,Taper_IN,Taper_out1,Taper_out2],
                    medium=clad,
                    symmetry=(0, 0, 1),  # symmetry in z
                    run_time = {'quality_factor' : 1, 'source_factor' : 3},
                    grid_spec=td.GridSpec.auto(wavelength = wavelength.min(),min_steps_per_wvl = 11),
                ))


                if os.path.exists(filename):
                    print(f"Loading {filename}")
                    filename_path = Path(filename)
                    sim_data_arr[thick_idx][length_idx].append(td.SimulationData.from_file(filename_path))

                else:
                    task_name = f"{version_name}_t{int(thick*1000)}_L{int(length_values*1000)}_W{int(width_values*1000)}"
                    job = web.Job(simulation= sim_arr[thick_idx][length_idx][width_idx], task_name=task_name)

                    print(f"Running simulation: {task_name}")

                    sim_data_arr[thick_idx][length_idx].append(job.run())
                    sim_data_arr[thick_idx][length_idx][width_idx].to_file(filename)

                    Job = web.Job(simulation= sim_arr[thick_idx][length_idx][width_idx], task_name="my_sim")

                    estimate+= Job.estimate_cost()

                    print(f"Estimated Maximum Cost: {estimate}")

            sim_data_arr[thick_idx].append([])
            sim_arr[thick_idx].append([])

    return sim_data_arr, sim_arr


